In [1]:
%pip install pymupdf torch transformers pillow

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install safetensors

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


In [4]:
import os
import fitz  # PyMuPDF
import torch
import io
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

# SETUP & CONFIGURATION
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

# Intercepting Hugging Face to prevent Jupyter crashes
try:
    import huggingface_hub.utils._progress
    import tqdm.std
    huggingface_hub.utils._progress.tqdm = tqdm.std.tqdm
except Exception:
    pass

PDF_FOLDER = "./Family Pic for Income verification/"               
CLEAN_IMAGE_FOLDER = "./01_RAW_IMAGES/"   
os.makedirs(PDF_FOLDER, exist_ok=True)
os.makedirs(CLEAN_IMAGE_FOLDER, exist_ok=True)

print("Loading CLIP Model for Usefulness Verification...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Hardware Acceleration: {device.upper()}")

# THE FIREWALL BYPASS: Notice the 'use_safetensors=True'
model_id = "./local_clip_model"
model = CLIPModel.from_pretrained(model_id).to(device)
processor = CLIPProcessor.from_pretrained(model_id)

CATEGORIES = [
    "a photograph of a house from the inside or outside, showing rooms, walls, or building exterior", 
    "a scanned document, blank page, text, chart, map, logo, or other unrelated items" 
]

# PIPELINE
def process_pdfs():
    print("\nStarting PDF Extraction and AI Filtering...")
    total_extracted = 0
    total_saved = 0
    
    for pdf_name in os.listdir(PDF_FOLDER):
        if not pdf_name.lower().endswith('.pdf'):
            continue
            
        pdf_path = os.path.join(PDF_FOLDER, pdf_name)
        base_name = os.path.splitext(pdf_name)[0]
        print(f"\nOpening PDF: {pdf_name}")
        
        try:
            doc = fitz.open(pdf_path)
            for page_num in range(len(doc)):
                image_list = doc[page_num].get_images(full=True)
                for img_index, img_info in enumerate(image_list):
                    total_extracted += 1
                    xref = img_info[0]
                    base_image = doc.extract_image(xref)
                    ext = base_image["ext"]
                    
                    try:
                        image = Image.open(io.BytesIO(base_image["image"])).convert("RGB")
                    except Exception:
                        continue 
                    
                    inputs = processor(text=CATEGORIES, images=image, return_tensors="pt", padding=True).to(device)
                    with torch.no_grad():
                        outputs = model(**inputs)
                        probs = outputs.logits_per_image.softmax(dim=-1)
                    
                    prob_target = probs[0][0].item()
                    if prob_target > 0.60:
                        save_name = f"{base_name}_pg{page_num+1}_img{img_index}.{ext}"
                        save_path = os.path.join(CLEAN_IMAGE_FOLDER, save_name)
                        image.save(save_path)
                        total_saved += 1
                        print(f"  [+] Kept: {save_name} (Confidence: {prob_target:.1%})")
                        
        except Exception as e:
            print(f"Error reading {pdf_name}: {e}")

    print("\nPIPELINE COMPLETE")
    print(f"Total raw images extracted: {total_extracted}")
    print(f"Total valid photos saved: {total_saved}")

if __name__ == "__main__":
    process_pdfs()

Loading CLIP Model for Usefulness Verification...
Hardware Acceleration: CPU


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


Starting PDF Extraction and AI Filtering...

Opening PDF: 000AC9DECAE48AC60C8B0ADC60CF432B55B18A06A5962C9F7ECEE7EBC50DE5CF.pdf
  [+] Kept: 000AC9DECAE48AC60C8B0ADC60CF432B55B18A06A5962C9F7ECEE7EBC50DE5CF_pg1_img0.jpeg (Confidence: 99.9%)

Opening PDF: 000FED5CA0C88C025D0CA20137B6BD17CF99D3870FC8A84190E507373585713C.pdf
  [+] Kept: 000FED5CA0C88C025D0CA20137B6BD17CF99D3870FC8A84190E507373585713C_pg1_img0.jpeg (Confidence: 100.0%)

Opening PDF: 00236E830A05174C10D0241D7AD2B50B840BD2734BA220356E84FD872537232A.pdf
  [+] Kept: 00236E830A05174C10D0241D7AD2B50B840BD2734BA220356E84FD872537232A_pg1_img0.jpeg (Confidence: 99.9%)

Opening PDF: 003286E1C17AAB3426FB6CE22B0FB1BD7A775090CA0C920923FDDBA6D69000B8.pdf

Opening PDF: 003D4B81EB86F50CE4CF43CF65B995256C0E60188344E486506B166BF83FCF4E.pdf

Opening PDF: 003E08FFCC0A29F172ED654A965B4DAA592D537F56CCBABEB24E10440CADB67B.pdf
  [+] Kept: 003E08FFCC0A29F172ED654A965B4DAA592D537F56CCBABEB24E10440CADB67B_pg1_img0.jpeg (Confidence: 76.1%)

Opening PDF

In [1]:
pip install opencv-python

   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.5/44.0 MB 1.8 MB/s eta 0:00:24
    --------------------------------------- 0.8/44.0 MB 2.1 MB/s eta 0:00:21
   - -------------------------------------- 1.3/44.0 MB 1.9 MB/s eta 0:00:23
   - -------------------------------------- 1.8/44.0 MB 2.0 MB/s eta 0:00:21
   - -------------------------------------- 2.1/44.0 MB 2.0 MB/s eta 0:00:21
   -- ------------------------------------- 2.4/44.0 MB 1.9 MB/s eta 0:00:22
   -- ------------------------------------- 2.9/44.0 MB 2.0 MB/s eta 0:00:21
   --- ------------------------------------ 3.4/44.0 MB 2.0 MB/s eta 0:00:21
   --- ------------------------------------ 3.9/44.0 MB 2.0 MB/s eta 0:00:20
   --- ------------------------------------ 4.2/44.0 MB 2.0 MB/s eta 0:00:21
   ---- ----------------------------------- 4.5/44.0 MB 2.0 MB/s eta 0:00:21
   ---- -----

In [2]:
import os
import shutil
import torch
import cv2
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

# Intercepting Hugging Face to prevent Jupyter crashes
try:
    import huggingface_hub.utils._progress
    import tqdm.std
    huggingface_hub.utils._progress.tqdm = tqdm.std.tqdm
except Exception:
    pass

RAW_IMAGES_FOLDER = "./01_RAW_IMAGES/"       
CLEAN_IMAGE_FOLDER = "./01_CLEAN_IMAGES/"   

os.makedirs(RAW_IMAGES_FOLDER, exist_ok=True)
os.makedirs(CLEAN_IMAGE_FOLDER, exist_ok=True)

# THRESHOLDS FOR PHYSICAL GATES
MIN_THUMBNAIL_SIZE = 128 # Drops anything smaller than 128x128. Keeps anything larger for padding.
BLUR_THRESHOLD = 50.0    # If the sharpness score is below this, it's considered blurry.

print("Loading CLIP Model OFFLINE from local folder...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Hardware Acceleration: {device.upper()}")

model_id = "./local_clip_model" 
model = CLIPModel.from_pretrained(model_id).to(device)
processor = CLIPProcessor.from_pretrained(model_id)

# ==========================================
# 2. UPDATED AI GATEKEEPER PROMPTS
# ==========================================
CATEGORIES = [
    "a photograph focusing on a single house or a specific room inside one house", # Index 0: TARGET
    "a photo collage containing multiple pictures stitched together in a grid",    # Index 1: DROP (Collages)
    "a wide street view showing multiple different houses, a village lane",        # Index 2: DROP (Multiple)
    "a scanned document, blank page, text, close-up texture, or unrelated items"   # Index 3: DROP (Junk)
]

VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')

# ==========================================
# 3. HELPER FUNCTION: BLUR DETECTION
# ==========================================
def check_sharpness(pil_img):
    """Converts image to grayscale and measures edge sharpness. Higher = Sharper."""
    # Convert PIL Image to an OpenCV Grayscale array
    cv_img = np.array(pil_img.convert('L'))
    # Calculate the variance of the Laplacian (standard blur detection math)
    sharpness_score = cv2.Laplacian(cv_img, cv2.CV_64F).var()
    return sharpness_score

# ==========================================
# 4. MAIN FILTER PIPELINE
# ==========================================
def process_images():
    print("\nStarting Advanced Image Filtering...")
    total_processed = 0
    total_saved = 0
    
    for image_name in os.listdir(RAW_IMAGES_FOLDER):
        
        if not image_name.lower().endswith(VALID_EXTENSIONS):
            continue
            
        image_path = os.path.join(RAW_IMAGES_FOLDER, image_name)
        total_processed += 1
        
        try:
            image = Image.open(image_path).convert("RGB")
            
            # --- GATE 1: THE SIZE CHECK ---
            width, height = image.size
            if width < MIN_THUMBNAIL_SIZE and height < MIN_THUMBNAIL_SIZE:
                print(f"  [-] Dropped (Thumbnail): {image_name} [{width}x{height}]")
                continue # Skip the rest of the checks to save time
                
            # --- GATE 2: THE BLUR CHECK ---
            sharpness = check_sharpness(image)
            if sharpness < BLUR_THRESHOLD:
                print(f"  [-] Dropped (Blurry): {image_name} [Score: {sharpness:.1f}]")
                continue # Skip the AI check to save GPU memory
            
            # --- GATE 3: THE AI CHECK (Collages & Content) ---
            inputs = processor(text=CATEGORIES, images=image, return_tensors="pt", padding=True).to(device)
            
            with torch.no_grad():
                outputs = model(**inputs)
                probs = outputs.logits_per_image.softmax(dim=-1)
            
            prob_single = probs[0][0].item()
            prob_collage = probs[0][1].item()
            prob_multiple = probs[0][2].item()
            prob_junk = probs[0][3].item()
            
            # It must be confident in the single house, AND it must beat all 3 junk categories
            if prob_single > 0.50 and prob_single > max(prob_collage, prob_multiple, prob_junk):
                save_path = os.path.join(CLEAN_IMAGE_FOLDER, image_name)
                shutil.copy(image_path, save_path)
                total_saved += 1
                print(f"  [+] Kept: {image_name} (Confidence: {prob_single:.1%}, Sharpness: {sharpness:.1f})")
            else:
                # Figure out exactly which junk category triggered the drop
                highest_junk = max(prob_collage, prob_multiple, prob_junk)
                if highest_junk == prob_collage:
                    reason = "Collage Detected"
                elif highest_junk == prob_multiple:
                    reason = "Multiple Houses"
                else:
                    reason = "Junk/Document"
                    
                print(f"  [-] Dropped ({reason}): {image_name}")
                    
        except Exception as e:
            print(f"Error reading {image_name}: {e}")

    print("\n" + "="*40)
    print("PIPELINE COMPLETE")
    print(f"Total raw images processed: {total_processed}")
    print(f"Total valid photos saved: {total_saved}")
    print(f"Removed {total_processed - total_saved} useless images!")
    print("="*40)

if __name__ == "__main__":
    process_images()

Loading CLIP Model OFFLINE from local folder...
Hardware Acceleration: CPU


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


Starting Advanced Image Filtering...
  [+] Kept: 000AC9DECAE48AC60C8B0ADC60CF432B55B18A06A5962C9F7ECEE7EBC50DE5CF_pg1_img0.jpeg (Confidence: 99.7%, Sharpness: 120.4)
  [-] Dropped (Blurry): 000FED5CA0C88C025D0CA20137B6BD17CF99D3870FC8A84190E507373585713C_pg1_img0.jpeg [Score: 31.6]
  [+] Kept: 00236E830A05174C10D0241D7AD2B50B840BD2734BA220356E84FD872537232A_pg1_img0.jpeg (Confidence: 99.4%, Sharpness: 727.9)
  [-] Dropped (Collage Detected): 003E08FFCC0A29F172ED654A965B4DAA592D537F56CCBABEB24E10440CADB67B_pg1_img0.jpeg
  [+] Kept: 007076CC85C2D341D92DEC5869E9A4E584545049614B9E9C534B6FFC480A1C0F_pg1_img0.jpeg (Confidence: 99.1%, Sharpness: 495.1)
  [+] Kept: 008CCE00006002901DAABE23F2A63CF018291970572FA3B5451EB742F0D41D8B_pg1_img0.jpeg (Confidence: 99.7%, Sharpness: 154.0)
  [-] Dropped (Collage Detected): 00DCC21F17AC94AC720BF9C4307A6BD11E909687EA54CF22ACCF610729B59B74_pg1_img0.jpeg
  [+] Kept: 0155DC5D2B8B35BEF52E296BC0C030645B404E358E8BB183C65121133E6FE14F_pg1_img0.jpeg (Confidence: